# Himawari-9 over Anak Krakatau - 1. Download historical HSD

Benny Istanto - https://benny.istan.to

---

The JMA real-time page only keeps the last 24 hours, which is why the
existing `himawari_animation.ipynb` cannot reach back to 4 September. This
notebook goes to the archive instead and pulls **Himawari Standard Data**
(HSD) - the calibrated Level 1b product, one file per band per scan segment,
at full native resolution.

**Two sources, same bytes.**

| | AWS `noaa-himawari9` | P-Tree `ftp.ptree.jaxa.jp` |
|---|---|---|
| Registration | none | required |
| Protocol | HTTPS, parallel | FTP, serial |
| Coverage | L1b full disk + some L2 | L1b, L2, gridded netCDF, the `/pub` geophysical products |

AWS is the default because the L1b objects are identical to P-Tree's
`/jma/hsd/` tree and it parallelises freely. Switch with `source.backend` in
`config.yml`; the P-Tree path is implemented and demonstrated at the bottom.

**Only the segments you need.** A full-disk scan is cut into 10 horizontal
strips. `catalog.segments_for_extent` solves the geostationary projection to
find which strips a lat/lon box falls into - for the Sunda Strait and the
eastern Indian Ocean that is segments 5 to 7, so the download is 30% of a
full disk rather than all of it.

## 1. Configuration

Everything below is driven by `config.yml`.

In [ ]:
import sys, warnings
from pathlib import Path

# The package lives in src/ and is not installed, so put it on the path.
ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

from himawari import project

cfg = project.load(ROOT / "config.yml")
cfg.make_dirs()
print(cfg.summary())

## 2. What is actually on the server

Two things routinely differ from a naive 10-minute grid, and both are worth
knowing *before* starting a long download rather than as a wall of 404s:

- AHI pauses full-disk scanning twice a day for housekeeping, at **02:40 and
  14:40 UTC**. `catalog.timeslots()` already skips those.
- The archive only extends to roughly the present moment, so an end time in
  the future silently yields nothing.

In [ ]:
from datetime import timedelta
from himawari.fetch import AwsBackend

probe = AwsBackend(cfg.satellite)

day = cfg.start.replace(hour=0, minute=0)
available = {}
while day <= cfg.end:
    slots = probe.available_slots(day)
    available[day.date()] = slots
    print(f"{day:%Y-%m-%d}  {len(slots):3d} slots"
          + (f"  {slots[0]} .. {slots[-1]} UTC" if slots else "  (nothing)"))
    day += timedelta(days=1)

wanted = cfg.timeslots()
present = [t for t in wanted if t.strftime("%H%M") in available.get(t.date(), [])]
print(f"\nrequested {len(wanted)} slots, {len(present)} of them exist on the server")
if len(present) < len(wanted):
    missing_after = [t for t in wanted if t not in present]
    print(f"first unavailable: {missing_after[0]:%Y-%m-%d %H:%M} UTC "
          f"(the archive has not caught up yet)")

## 3. How big is this going to be

Sizes are measured, not guessed: one `ListObjectsV2` call per sample timeslot
returns the exact `Content-Length` of every object in it.

Sampling several slots spread across the window matters more than it looks
like it should. bz2 squeezes a night-time visible band - a nearly uniform dark
field - to a fraction of its daylight size, so estimating true colour from a
single midnight sample understates a three-day download by about half.

In [ ]:
import shutil
from himawari import catalog
from himawari.fetch import estimate

tasks = catalog.frame_tasks(present, cfg.bands, cfg.segments(), cfg.satellite)

est = estimate(tasks, probe, sample_slots=9)
free_gb = shutil.disk_usage(cfg.cache_dir.anchor).free / 2**30

print(f"bands       {' '.join(cfg.bands)}")
print(f"segments    {cfg.segments()}")
print(f"files       {est['files']:,}")
print(f"per slot    {est['mb_per_slot']:.0f} MB "
      f"(night {est['mb_per_slot_min']:.0f} - day {est['mb_per_slot_max']:.0f})")
print(f"total       {est['total_gb']:.1f} GB")
print(f"free disk   {free_gb:.0f} GB on {cfg.cache_dir.anchor}")
print("OK" if free_gb > est['total_gb'] * 1.2 else "NOT ENOUGH ROOM - narrow the window or drop a composite")

### If that is more than you want

The cost is dominated by band 3, the 0.5 km red channel that true colour needs
for its sharpening: on its own it is roughly half the total. Options, in order
of how much they save:

- drop `true_color` from `products` in `config.yml` and keep the IR composites,
  which work day and night - about 12 GB for the same three days
- coarsen `window.step_minutes` to 20 or 30
- shorten the window

`cfg.bands` is derived from the product list, so removing a composite really
does shrink the download.

## 4. Download

Resumable by construction. Each file streams to `<name>.part` and is renamed
only once the byte count matches `Content-Length`, so an interrupted run leaves
no half-written file that a later run would mistake for good data - just
re-run this cell.

Work is ordered time-major, so an interruption leaves whole usable scenes
behind rather than one band smeared across the whole period.

In [ ]:
from himawari.fetch import download

backend = cfg.backend()
report = download(
    tasks,
    cfg.cache_dir,
    backend=backend,
    workers=cfg.download_workers,
)
print(report.summary())

manifest = report.to_csv(cfg.temp_dir / "download_manifest.csv")
print("manifest:", manifest)

## 5. Check what arrived

A scene missing one segment renders as a frame with a blank stripe across it,
which is worse than a dropped frame - so notebook 02 only renders timeslots
that are complete. This is where you find out which ones those are.

In [ ]:
from himawari.fetch import complete_times

good, bad = complete_times(tasks, cfg.cache_dir)
print(f"complete timeslots   {len(good)}")
print(f"incomplete           {len(bad)}")
for t in bad[:10]:
    print("   ", t.strftime("%Y-%m-%d %H:%M UTC"))
if len(bad) > 10:
    print(f"    ... and {len(bad) - 10} more")

for o in report.failures()[:10]:
    print(f"  {o.status:8s} {o.task.filename}  {o.detail}")

### Optional: prove the archives are intact

`is_cached` only checks that a file exists and is not obviously truncated,
because it runs once per file per notebook run. The thorough check decompresses
every byte. It is slow, so it is opt-in - but it is worth running once over a
finished cache, because satpy's failure mode on a corrupt bz2 member is an
opaque struct error 300 frames into the render.

In [ ]:
import random
from himawari.fetch import verify_bz2

sample = random.sample(tasks, min(40, len(tasks)))
bad_files = [t.filename for t in sample
             if not verify_bz2(t.path(cfg.cache_dir))]
print(f"checked {len(sample)} files, {len(bad_files)} corrupt")
for f in bad_files:
    print("   ", f)

# To check everything instead (slow):
# bad_files = [t.filename for t in tasks if not verify_bz2(t.path(cfg.cache_dir))]

## 6. Using P-Tree instead

Set `source.backend: ptree` in `config.yml` and put your registered
credentials in `.env` (copy `.env.example`). They are read from the
environment, never written into a cell or the manifest.

```
P_TREE_USER=your_registered_email
P_TREE_PASS=your_password
```

Register at <https://www.eorc.jaxa.jp/ptree/registration_top.html>.

P-Tree shelves the same files differently - `/jma/hsd/YYYYMM/DD/HH/` rather
than one directory per 10-minute slot - which `catalog.ptree_path` handles.
Worker count is capped at 4 regardless of `source.workers`: FTP throughput
collapses past a handful of connections, and P-Tree is a shared academic
service rather than a CDN.

`/jma/` is the Standard Data tree used here. `/pub/` holds the geophysical
parameter products - cloud properties, aerosol, SST - which have no AWS
equivalent and are the real reason to keep a P-Tree account.

In [ ]:
# Uncomment to fetch a single timeslot over FTP as a connectivity test.
#
# from himawari.fetch import PTreeBackend, download
# from himawari import catalog
#
# ptree = PTreeBackend()                       # reads P_TREE_USER / P_TREE_PASS
# one = catalog.frame_tasks(present[:1], ["B13"], cfg.segments(), cfg.satellite)
# print(one[0].ftp_path)
# print(download(one, cfg.cache_dir, backend=ptree, workers=2).summary())

---

Next: **`02_process.ipynb`** turns this cache into map-projected frames.